In [ ]:
# External Imports
import sys
from pathlib import Path
import torch
import torch.optim as optim
import torch.nn as nn

# Internal Imports
sys.path.insert(0, '../src')
from src.Dataset.mri_split import split_patients
from src.Dataset.data_loaders import get_dataloaders
from src.Dataset.cache import load_cache
from src.Model.train import fit
from src.Model.Unet.Unet import UNetModel, DiceLoss

In [ ]:
accepted_data = {}
rejected_data = {}
try:
    accepted_data = load_cache(Path("../data/processed/cache/accepted_data.json"), Path.cwd().parent)
    rejected_data = load_cache(Path("../data/processed/cache/rejected_data.json"), Path.cwd().parent)
except BaseException as e:
    print(e)

In [ ]:
SPLIT_SEED = 42
train_patients, val_patients, test_patients = split_patients(accepted_data, seed=SPLIT_SEED)

In [ ]:
BATCH_SIZE = 32
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(
    train_patients,
    val_patients,
    test_patients,
    batch_size=BATCH_SIZE,
    segmentation=True
)

In [ ]:
images, masks = next(iter(train_dataloader))

In [ ]:
print(images.shape)
print(masks.shape)
print(masks.unique())
print(images.dtype)
print(masks.dtype)

# CHECKPOINT

Next is writing the training loop

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(device)

model = UNetModel(in_channels = 3, out_channels = 1)
model = model.to(device)

loss_fn = DiceLoss()

OPTIMIZER = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 10
PATIENCE = 5
SEGMENTATION = True

In [ ]:
fit(model, train_dataloader, val_dataloader, loss_fn, OPTIMIZER, device, EPOCHS, PATIENCE, SEGMENTATION)

In [ ]:
from PIL import Image
img = Image.open("/Users/codymacdonald/Documents/GitHub/BrainTumorMRIClassification/data/raw/lgg-mri-segmentation/kaggle_3m/TCGA_CS_4941_19960909/TCGA_CS_4941_19960909_19_mask.tif")

In [ ]:
img.size